# Sample 20% of final annotation data

Samples 20% of rows from every CSV under `data/annotations/Final_Annotations/` and writes them to `data/annotations/Final_Annotations_sample_20/`, mirroring the original folder structure.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
FRACTION = 0.20

ROOT = Path.cwd().parents[1]  # repo root (notebook lives in src/validation/)
SRC_DIR = ROOT / "data" / "annotations" / "Final_Annotations"
DST_DIR = ROOT / "data" / "annotations" / "Final_Annotations_sample_20"

print(f"Source: {SRC_DIR}")
print(f"Destination: {DST_DIR}")

In [ ]:
csv_paths = sorted(SRC_DIR.rglob("*.csv"))
print(f"Found {len(csv_paths)} CSV files:")
for p in csv_paths:
    print(f"  {p.relative_to(SRC_DIR)}")

## Sample and save

Each CSV is sampled independently — 20% of its rows (at least 1), drawn without replacement with a fixed seed for reproducibility.

In [ ]:
rng = np.random.default_rng(SEED)
summary = []

for csv_path in csv_paths:
    rel = csv_path.relative_to(SRC_DIR)  # e.g. IDS/Regular_video_comments_Cleared.csv

    df = pd.read_csv(csv_path)

    n_sample = max(1, round(len(df) * FRACTION))
    idx = np.sort(rng.choice(len(df), size=n_sample, replace=False))
    sample = df.iloc[idx]

    out_path = DST_DIR / rel
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sample.to_csv(out_path, index=False)

    summary.append({
        "file": str(rel),
        "total_rows": len(df),
        "sampled_rows": len(sample),
    })

summary_df = pd.DataFrame(summary)
summary_df

## Verify

Check the output mirrors the source structure and that each file has the expected sample size.

In [ ]:
out_paths = sorted(DST_DIR.rglob("*.csv"))
assert {p.relative_to(DST_DIR) for p in out_paths} == {p.relative_to(SRC_DIR) for p in csv_paths}
print(f"OK — {len(out_paths)} files written, structure matches.\n")

# Check sample sizes match expectation
for row in summary:
    written = pd.read_csv(DST_DIR / row["file"])
    expected = max(1, round(row["total_rows"] * FRACTION))
    assert len(written) == expected == row["sampled_rows"], f"Size mismatch: {row['file']}"
print("OK — all sample sizes are 20% of the source rows.")